In [ ]:
# Solar Energy Generation & Economic Viability
# Estimate kWh/year and Payback Period per building

In [9]:
print("Running economic model...")

# Parameters
kwh_per_kw = 1280
discount_rate = 0.085
lifetime = 25
degradation_rate = 0.006
self_consumption_rate = 0.65

# System sizing
buildings['system_kw'] = np.minimum(buildings['usable_area_m2'] / 6.5, 35)

# Year 1 energy
buildings['annual_kwh_year1'] = buildings['system_kw'] * kwh_per_kw

# Installation cost
fixed_cost = 12500
variable_cost_per_kw = 7250
buildings['installation_cost_ghs'] = fixed_cost + (buildings['system_kw'] * variable_cost_per_kw)
buildings['installation_cost_ghs'] = np.where(
    buildings['system_kw'] > 15,
    buildings['installation_cost_ghs'] * 0.90,
    buildings['installation_cost_ghs']
)

# O&M
buildings['annual_om_ghs'] = buildings['system_kw'] * 65

# Tariff
buildings['effective_tariff_ghs_kwh'] = 2.15

# Vectorized NPV
print("Calculating NPV...")

years = np.arange(1, lifetime + 1)
discount_factors = 1 / ((1 + discount_rate) ** years)
degradation_factors = (1 - degradation_rate) ** (years - 1)
combined_factor = discount_factors * degradation_factors

# Initialize NPV array
npv = np.zeros(len(buildings))

# Loop over years
for t in range(lifetime):
    factor = combined_factor[t]
    
    energy_t = buildings['annual_kwh_year1'] * factor
    
    savings_t = (
        energy_t * self_consumption_rate * buildings['effective_tariff_ghs_kwh']
        - buildings['annual_om_ghs']
    )
    
    npv += savings_t

buildings['npv_ghs'] = npv - buildings['installation_cost_ghs']

# Simple payback for comparison
buildings['payback_years'] = buildings['installation_cost_ghs'] / (
    (buildings['annual_kwh_year1'] * self_consumption_rate * buildings['effective_tariff_ghs_kwh']) 
    - buildings['annual_om_ghs']
)
buildings['payback_years'] = buildings['payback_years'].clip(upper=25)

# Summary 
print("\n=== ECONOMIC SUMMARY ===")
print(buildings[['system_kw', 'annual_kwh_year1', 'installation_cost_ghs',
                 'payback_years', 'npv_ghs']].describe().round(0))

print(f"\nTotal potential Year-1 generation : {buildings['annual_kwh_year1'].sum() / 1_000_000:.1f} GWh/year")
print(f"Average simple payback             : {buildings['payback_years'].mean():.1f} years")
print(f"Average NPV                        : {buildings['npv_ghs'].mean():,.0f} GHS")
print(f"Buildings with positive NPV        : {(buildings['npv_ghs'] > 0).sum():,}")
print(f"Buildings with payback 5–10 years  : {((buildings['payback_years'] >= 5) & (buildings['payback_years'] <= 10)).sum():,}")

Running economic model...
Calculating NPV...

=== ECONOMIC SUMMARY ===
       system_kw  annual_kwh_year1  installation_cost_ghs  payback_years  \
count   630946.0          630946.0               630946.0       630946.0   
mean         8.0            9703.0                64416.0            7.0   
std          8.0           10801.0                54696.0            3.0   
min          0.0             221.0                13751.0            4.0   
25%          2.0            2302.0                25541.0            5.0   
50%          4.0            4982.0                40720.0            6.0   
75%         10.0           13155.0                87011.0            8.0   
max         35.0           44800.0               239625.0           25.0   

        npv_ghs  
count  630946.0  
mean    55594.0  
std     79032.0  
min    -11020.0  
25%      2937.0  
50%     20906.0  
75%     75702.0  
max    314498.0  

Total potential Year-1 generation : 6121.8 GWh/year
Average simple payback       

In [10]:
# Save

# Ensure building_id exists
if 'building_id' not in buildings.columns:
    buildings['building_id'] = range(1, len(buildings) + 1)

# Main comprehensive file
final_gpkg = "accra_buildings_solar_roi_final.gpkg"
buildings.to_file(final_gpkg, driver="GPKG")
print(f"Full final dataset saved as: {final_gpkg}")

# Clean summary CSV
summary_cols = ['building_id', 'usable_area_m2', 'slope', 'aspect', 
                'suitability_score', 'system_kw', 'annual_kwh_year1', 
                'installation_cost_ghs', 'annual_savings_ghs', 
                'payback_years', 'npv_ghs']

buildings[summary_cols].to_csv("accra_solar_final_summary.csv", index=False)
print("   - accra_solar_final_summary.csv (ready for visualization)")


print(f"Total buildings analyzed       : {len(buildings):,}")
print(f"Buildings with positive NPV    : {515007:,} ({515007/len(buildings)*100:.1f}%)")
print(f"Average payback period         : {buildings['payback_years'].mean():.1f} years")
print(f"Average NPV per building       : +{buildings['npv_ghs'].mean():,.0f} GHS")

Full final dataset saved as: accra_buildings_solar_roi_final.gpkg
   - accra_solar_final_summary.csv (ready for visualization)
Total buildings analyzed       : 632,195
Buildings with positive NPV    : 515,007 (81.5%)
Average payback period         : 7.0 years
Average NPV per building       : +55,594 GHS
